# Sepid Island RAG — Colab entry point
This notebook uses the same package that later runs in Docker. It contains orchestration only, not a second implementation.

## 1. Upload and unpack the delivered ZIP
Run this cell, choose `sepid-rag-backend.zip`, and then install the project.

In [ ]:
from google.colab import files
uploaded = files.upload()
archive = next(iter(uploaded))
!unzip -q -o "$archive" -d /content
%cd /content/sepid-rag-backend
!pip -q install -e .

## 2. Verify the package with development-only components
These results test wiring only and must not be reported as embedding quality.

In [ ]:
import os
os.environ.update({
    'APP_ENV': 'development',
    'EMBEDDING_PROVIDER': 'hashing',
    'EMBEDDING_MODEL': 'development-only',
    'EMBEDDING_REVISION': 'development-only',
    'EMBEDDING_DIMENSION': '768',
    'QUERY_PREFIX': '',
    'DOCUMENT_PREFIX': '',
    'TOP_K': '5',
    'LLM_PROVIDER': 'echo',
    'LLM_MODEL': 'development-only',
})
!python scripts/check_corpus.py
!python scripts/smoke_chat.py 'هتل صدف چقدر هزینه دارد؟'

## 3. Evaluate local multilingual E5
This downloads and runs the open-source embedding model in Colab. It requires no paid API. The Hugging Face commit is resolved and recorded so the evaluation is reproducible.

In [ ]:
!pip -q install -e ".[local-embeddings]"
from huggingface_hub import model_info
embedding_model = 'intfloat/multilingual-e5-large-instruct'
embedding_revision = model_info(embedding_model).sha
print('Pinned embedding revision:', embedding_revision)
os.environ['APP_ENV'] = 'evaluation'
os.environ['EMBEDDING_PROVIDER'] = 'sentence_transformers'
os.environ['EMBEDDING_MODEL'] = embedding_model
os.environ['EMBEDDING_REVISION'] = embedding_revision
os.environ['EMBEDDING_DIMENSION'] = '1024'
os.environ['QUERY_PREFIX'] = 'Instruct: Given a Persian search query, retrieve relevant Persian passages that answer the query\nQuery: '
os.environ['DOCUMENT_PREFIX'] = ''
os.environ['TOP_K'] = '5'  # candidate, not final
!python scripts/evaluate_retrieval.py --output reports/e5_k5.json

## 4. Test Groq chat only after inspecting retrieval
Store the key in Colab Secrets as `GROQ_API_KEY`. The chat model below is a candidate and must be tested in Persian before it is frozen.

In [ ]:
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
os.environ['LLM_PROVIDER'] = 'groq'
os.environ['LLM_MODEL'] = 'qwen/qwen3.6-27b'
os.environ['APP_ENV'] = 'experiment'
!python scripts/smoke_chat.py 'در بازار محله صدف چگونه خرید کنیم؟'